In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import setup_ai
import langchain_community
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb


In [2]:
# Initialize the model using our central config
model = setup_ai()

[INFO] Git protection active (.env is hidden).
[INFO] Testing connection to gemini-3-flash-preview...
[SUCCESS] Connected to gemini-3-flash-preview


In [4]:
# import os
# from dotenv import load_dotenv
# from google import genai

# # 1. Load your .env file
# load_dotenv()
# api_key = os.getenv("GEMINI_API_KEY")

# # 2. Create the Client directly
# # The SDK automatically uses your GEMINI_API_KEY if it's in your environment
# client = genai.Client(api_key=api_key)

# # 3. Test the connection with the best current model
# # Model: gemini-3-flash-preview (The 2026 standard for speed and intelligence)
# response = client.models.generate_content(
#     model='gemini-3-flash-preview', 
#     contents="Confirm connection: What is the capital of Moldova?"
# )

# print("Connection check successful.")
# print("Model output:", response.text)

In [5]:
# 1. Re-connect to the existing folder (Automatic)
client = chromadb.PersistentClient(path="./my_vectordb")

# 2. Get the existing collection (Automatic)
# 'get_or_create' means: "If it exists, just open it. Don't make a new one."
collection = client.get_or_create_collection(name="user_knowledge")

print("Database is initialized and ready!")

Database is initialized and ready!


In [6]:
def prepare_data(file_path):
    """
    Loads a file (PDF or Text) and splits it into manageable chunks for the AI's long-term memory.
    """

    # 1. Determine file type and load the comtent
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)
    
    # This reads the file and creates a list of 'Document' objects
    raw_documents = loader.load()

    # 2. Configure the text splitter
    # chunk_size: How many characters in each block
    # chunk_overlap: How many characters to repeat from the previous block
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50,
        add_start_index=True,
        separators=["\n\n", "\n", " ", ""]
    )

    # 3. Perform the split
    chunks = text_splitter.split_documents(raw_documents)

    print(f"Data Audit: File split into {len(chunks)} chunks.")

    return chunks

# Example of how you will call this function:
# my_chunks = prepare_data("my_notes.txt")

In [7]:
# 1. Run the function on test file
my_chunks = prepare_data("../my_knowledge/travel.txt")

# 2. Check the "type" to ensure it returned a list
print(f"Data type: {type(my_chunks)}")

# 3. Look at the very first chunk to see if it looks right
if len(my_chunks) > 0:
    print("---First Chunk Content---")
    print(my_chunks[0].page_content)
    print("-------------------------")
    print(f"Metadata: {my_chunks[0].metadata}")

Data Audit: File split into 4 chunks.
Data type: <class 'list'>
---First Chunk Content---
# TRAVEL NOTES: Denmark & Beyond (2025-2026)
-------------------------
Metadata: {'source': '../my_knowledge/travel.txt', 'start_index': 0}


In [8]:
# 1. Grab the text from the first chunk
text_sample = my_chunks[0].page_content

# 2. Using the client to get the math
response = model.models.embed_content(model='text-embedding-004', contents=text_sample)

In [9]:
# Dig into the response to get actual numbers
vector = response.embeddings[0].values

print("Success! Your text is now math.")
print(f"First 5 numbers of the vector: {vector[:5]}")
print(f"Total length of the vector: {len(vector)}")

Success! Your text is now math.
First 5 numbers of the vector: [0.03107574, 0.006146332, -0.034929197, -0.034178715, -0.004060326]
Total length of the vector: 768


In [10]:
query_text = "What should I eat for dinner?"

In [11]:
query_vector = model.models.embed_content(model='text-embedding-004', contents=query_text)

In [12]:
len(query_vector.embeddings[0].values)

768

In [13]:
all_scores = []

In [14]:
query_vector_values = query_vector.embeddings[0].values

for v in my_chunks:
    text_sample = v.page_content
    response = model.models.embed_content(model='text-embedding-004', contents=text_sample)
    vector = response.embeddings[0].values
    current_score = cosine_similarity([query_vector_values], [vector])[0][0]
    all_scores.append(current_score)

print("Results for each chunk:", all_scores)

Results for each chunk: [np.float64(0.23910434581818724), np.float64(0.30780336260875385), np.float64(0.265918813183442), np.float64(0.29974145462530033)]


In [15]:
best_index = np.argmax(all_scores)
print(f"The AI found the best match for query: {query_text} and it is:")
print(my_chunks[best_index].page_content)

The AI found the best match for query: What should I eat for dinner? and it is:
## Move Preparation: Denmark
* Geography: Denmark is the southernmost of the Scandinavian countries. It consists of the Jutland peninsula and over 400 named islands.
* Transport: Get the "Rejsekort" (travel card) immediately upon arrival in Copenhagen. It works for buses, trains, and the metro.
* Housing Tip: Many Danish apartments are rented without light fixtures in the ceiling. I need to remember to buy lamps or "pendel" lights.


In [16]:
my_vectors = []

for v in my_chunks:
    response = model.models.embed_content(model='text-embedding-004', contents=v.page_content)
    my_vectors.append(response.embeddings[0].values)


def find_similarity(query_text, chunks, vectors):
    query_vector = model.models.embed_content(model='text-embedding-004', contents=query_text)
    query_vector_values = query_vector.embeddings[0].values
    all_scores = []
    for v in vectors:
        current_score = cosine_similarity([query_vector_values], [v])[0][0]
        all_scores.append(current_score)

    best_index = np.argmax(all_scores)
    best_score = np.max(all_scores)
    if best_score >= 0.4:
        return f"The AI found the best match for query: {query_text} and it is: {my_chunks[best_index].page_content}"
    else:
        return "I'm sorry, I don't have information about that in my database."

In [17]:
query_text_test = "What is the syntax for repeating a task in a programming language?"

In [18]:
find_similarity(query_text_test, my_chunks, my_vectors)

"I'm sorry, I don't have information about that in my database."